# IBM Telco Customer Churn — Data Understanding

This notebook inspects the raw Telco customer data before modeling. It reviews the target, feature types, missing values, duplicates, quality checks, and common churn patterns. It does not modify the raw CSV.

Expected dataset path: `raw/dataset.csv`. Install the project EDA dependencies first with `python -m pip install -r requirements-eda.txt`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
data_path = project_root / "raw" / "dataset.csv"
if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found: {data_path}")

raw_df = pd.read_csv(data_path)
df = raw_df.copy()
print(f"Dataset: {data_path}")
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
display(df.head())

## Target and feature types

`Churn` is the binary target: `Yes` means the customer left and `No` means the customer stayed. `customerID` is an identifier, not a predictive feature. `SeniorCitizen` is stored as 0/1 and is best interpreted as a binary category. `TotalCharges` is delivered as text in this dataset and must be converted to numeric.

In [ ]:
print("Column data types as loaded:")
display(df.dtypes.rename("dtype").to_frame())

target = df["Churn"].astype("string").str.strip()
target_counts = target.value_counts(dropna=False).rename_axis("Churn").to_frame("customers")
target_counts["share"] = target_counts["customers"] / len(df)
display(target_counts)
print(f"Overall churn rate: {(target == 'Yes').mean():.1%}")

numeric_columns = df.select_dtypes(include="number").columns.tolist()
categorical_columns = df.select_dtypes(include=["object", "string", "category"]).columns.tolist()
print("Numeric columns (as loaded):", numeric_columns)
print("Categorical/text columns:", categorical_columns)
print("Identifier to exclude from modeling: customerID")

## Missing values, duplicates, and data quality

Blank `TotalCharges` entries can look like ordinary strings rather than nulls, so this section checks both raw nulls and nulls after numeric conversion. It also checks duplicate rows, repeated customer IDs, target labels, and basic numeric ranges.

In [ ]:
quality_df = df.copy()
raw_total = quality_df["TotalCharges"].astype("string").str.strip()
blank_total_charges = raw_total.eq("").sum()
quality_df["TotalCharges"] = pd.to_numeric(raw_total.replace("", pd.NA), errors="coerce")
missing = quality_df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print(f"Exact duplicate rows: {quality_df.duplicated().sum():,}")
print(f"Duplicate customer IDs: {quality_df['customerID'].duplicated().sum():,}")
print(f"Blank TotalCharges strings before conversion: {blank_total_charges:,}")
print("Missing values after TotalCharges conversion:")
display(missing.to_frame("missing_count"))

valid_targets = {"Yes", "No"}
unknown_targets = sorted(set(target.dropna().unique()) - valid_targets)
print("Unexpected Churn labels:", unknown_targets)
range_checks = {
    "negative tenure": int((quality_df["tenure"] < 0).sum()),
    "tenure above 72 months": int((quality_df["tenure"] > 72).sum()),
    "negative MonthlyCharges": int((quality_df["MonthlyCharges"] < 0).sum()),
    "negative TotalCharges": int((quality_df["TotalCharges"] < 0).sum()),
}
display(pd.Series(range_checks, name="rows").to_frame())

print("Data quality notes:")
print("- Convert TotalCharges to numeric; blank entries become missing and should be imputed from training data.")
print("- Drop customerID before model fitting because it is an identifier.")
print("- Keep all imputers, encoders, and scalers inside a pipeline fitted only on training data.")

## Churn patterns

The next cells compare churn rates across contract, internet service, payment method, and tenure groups. These are descriptive associations, not evidence that a feature causes churn.

In [ ]:
analysis_df = quality_df.copy()
analysis_df["ChurnFlag"] = target.map({"No": 0, "Yes": 1})
analysis_df["tenure_group"] = pd.cut(
    analysis_df["tenure"], bins=[-1, 12, 24, 48, 72],
    labels=["0–12 months", "13–24 months", "25–48 months", "49–72 months"]
)

def churn_rate_by(column):
    return (
        analysis_df.groupby(column, observed=False)["ChurnFlag"]
        .agg(customers="size", churn_rate="mean")
        .sort_values("churn_rate", ascending=False)
    )

for column in ["Contract", "InternetService", "PaymentMethod", "tenure_group"]:
    print(f"Churn rate by {column}:")
    display(churn_rate_by(column).style.format({"churn_rate": "{:.1%}"}))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, column in zip(axes.flat, ["Contract", "InternetService", "PaymentMethod", "tenure_group"]):
    plot_data = churn_rate_by(column).reset_index()
    sns.barplot(data=plot_data, x=column, y="churn_rate", ax=ax, color="#4C78A8")
    ax.set_title(f"Churn rate by {column}")
    ax.set_ylabel("Churn rate")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=25)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0%}"))
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.boxplot(data=analysis_df, x="Churn", y="MonthlyCharges", ax=axes[0])
axes[0].set_title("Monthly charges by churn outcome")
sns.boxplot(data=analysis_df, x="Churn", y="tenure", ax=axes[1])
axes[1].set_title("Tenure by churn outcome")
fig.tight_layout()
plt.show()

summary = analysis_df.groupby("Churn").agg(
    customers=("ChurnFlag", "size"),
    mean_tenure=("tenure", "mean"),
    mean_monthly_charges=("MonthlyCharges", "mean"),
)
display(summary.round(2))

## Interpretation

For this CSV (7,043 customers), overall churn is 26.5%; there are 11 blank `TotalCharges` values, no exact duplicate rows, and no repeated customer IDs. Churn is 42.7% for month-to-month contracts versus 11.3% for one-year and 2.8% for two-year contracts. It is 47.4% for customers with up to 12 months of tenure and 9.5% for 49–72 months. Fiber-optic customers show 41.9% churn, and electronic-check customers 45.3%. Customers who churn have higher average monthly charges in this data ($74.44 versus $61.27). These are descriptive associations, not causal effects.